In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split 
import time
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
import pickle
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier   
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier

In [3]:
dataset1=pd.read_csv("Device_Monitering.csv",index_col=None)

In [4]:
dataset1.drop(dataset1.columns[5:198],axis=1,inplace=True)

In [5]:
dataset1.columns

Index(['date', 'serial_number', 'model', 'capacity_bytes', 'failure'], dtype='object')

In [6]:
dataset1=dataset1.drop('date',axis=1)

In [7]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
dataset1['serial_number'] = le.fit_transform(dataset1['serial_number']).astype(float)
dataset1['model'] = le.fit_transform(dataset1['model']).astype(float)

In [8]:
dataset1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 331733 entries, 0 to 331732
Data columns (total 4 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   serial_number   331733 non-null  float64
 1   model           331733 non-null  float64
 2   capacity_bytes  331733 non-null  float64
 3   failure         331733 non-null  int64  
dtypes: float64(3), int64(1)
memory usage: 10.1 MB


In [9]:
dataset1.isnull().sum()

serial_number     0
model             0
capacity_bytes    0
failure           0
dtype: int64

In [10]:
indep_X=dataset1.drop(columns=['failure'])
dep_Y=dataset1['failure']

In [11]:
def selectkbest(indep_X,dep_Y,n):
    test =SelectKBest(score_func=chi2,k=n)
    fit1=test.fit(indep_X,dep_Y)
    selectk_features=fit1.transform(indep_X)
    return selectk_features
def split_scaler(indep_X,dep_Y):
    X_train,X_test,Y_train,Y_test=train_test_split(indep_X,dep_Y,test_size=0.25,random_state=0)
    sc=StandardScaler()
    X_train=sc.fit_transform(X_train)
    X_test=sc.transform(X_test)
    return X_train,X_test,Y_train,Y_test
    
def cm_predicition(classifier,X_test):
    Y_pred=classifier.predict(X_test)

    from sklearn.metrics import confusion_matrix
    cm=confusion_matrix(Y_test,Y_pred)

    from sklearn.metrics import accuracy_score
    Accuracy = accuracy_score(Y_test,Y_pred)
    
    from sklearn.metrics import classification_report
    report=classification_report(Y_test,Y_pred)

    return classifier,Accuracy,report,X_test,Y_test,cm
    
def logistic(X_train,Y_train,X_test):
    from sklearn.linear_model import LogisticRegression
    classifier =LogisticRegression(random_state=0)
    classifier.fit(X_train,Y_train)
    classifier,Accuracy,report,X_test,Y_test,cm=cm_predicition(classifier,X_test)
    return classifier,Accuracy,report,X_test,Y_test,cm
    
def svm_linear(X_train,Y_train,X_test):
    from sklearn.svm import SVC
    classifier =SVC(kernel='linear',random_state =0)
    classifier.fit(X_train,Y_train)
    classifier,Accuracy,report,X_test,Y_test,cm=cm_predicition(classifier,X_test)
    return classifier,Accuracy, report,X_test,Y_test,cm
    
def svm_NL(X_train,Y_train,X_test):
    from sklearn.svm import SVC
    classifier =SVC(kernel='linear',random_state=0)
    classifier.fit(X_train,Y_train)
    classifier,Accuracy,report,X_test,Y_test,cm=cm_predicition(classifier,X_test)
    return classifier,Accuracy, report,X_test,Y_test,cm
    
def Navie(X_train,Y_train,X_test):
    from sklearn.naive_bayes import GaussianNB
    classifier = GaussianNB()
    classifier.fit(X_train,Y_train)
    classifier,Accuracy,report,X_test,Y_test,cm=cm_predicition(classifier,X_test)
    return classifier,Accuracy, report,X_test,Y_test,cm
    
def knn(X_train,Y_train,X_test):
    from sklearn.neighbors import KNeighborsClassifier
    classifier =KNeighborsClassifier(n_neighbors=5,metric='minkowski',p=2)
    classifier.fit(X_train,Y_train)
    classifier,Accuracy,report,X_test,Y_test,cm=cm_predicition(classifier,X_test)
    return classifier,Accuracy, report,X_test,Y_test,cm
    
def Decision(X_train,Y_train,X_test):
    from sklearn.tree import DecisionTreeClassifier
    classifier =DecisionTreeClassifier(criterion ='entropy',random_state=0)
    classifier.fit(X_train,Y_train)
    classifier,Accuracy,report,X_test,Y_test,cm=cm_predicition(classifier,X_test)
    return classifier,Accuracy, report,X_test,Y_test,cm
    
def random(X_train,Y_train,X_test):
    from sklearn.ensemble import RandomForestClassifier
    classifier =RandomForestClassifier(n_estimators =10,criterion='entropy',random_state=0)
    classifier.fit(X_train,Y_train)
    classifier,Accuracy,report,X_test,Y_test,cm=cm_predicition(classifier,X_test)
    return classifier,Accuracy, report,X_test,Y_test,cm
    
def selectk_Classification(acclog,accsvml,accsvmnl,accknn,accnav,accdes,accrf):
    dataframe=pd.DataFrame(index=['ChiSquare'],columns=['Logistic','SVMl','SVMnl','KNN','Navie',
                                                        'Decision','Random'])
    for number,ind in enumerate(dataframe.index):
        dataframe['Logistic'][ind]=acclog[number] 
        dataframe['SVMl'][ind]=accsvml[number]
        dataframe['SVMnl'][ind]=accsvmnl[number]
        dataframe['KNN'][ind]=accknn[number]
        dataframe['Navie'][ind]=accnav[number]
        dataframe['Decision'][ind]=accdes[number]
        dataframe['Random'][ind]=accrf[number]
    return dataframe

In [12]:
dataset1

,serial_number,model,capacity_bytes,failure
0,47529.0,0.0,2.500590e+11,0
1,47584.0,0.0,2.500590e+11,0
2,49278.0,0.0,2.500590e+11,0
3,49279.0,0.0,2.500590e+11,0
4,50487.0,0.0,2.500590e+11,0
...,...,...,...,...
331728,96717.0,75.0,2.200100e+13,0
331729,96724.0,75.0,2.200100e+13,0
331730,243257.0,75.0,2.200100e+13,0
331731,243294.0,75.0,2.200100e+13,0


In [16]:
kbest = selectkbest(indep_X,dep_Y,3)
acclog=[]
accsvml=[]
accsvmnl=[]
accknn=[]
accnav=[]
accdes=[]
accrf=[]

In [17]:
kbest

array([[4.75290e+04, 0.00000e+00, 2.50059e+11],
       [4.75840e+04, 0.00000e+00, 2.50059e+11],
       [4.92780e+04, 0.00000e+00, 2.50059e+11],
       ...,
       [2.43257e+05, 7.50000e+01, 2.20010e+13],
       [2.43294e+05, 7.50000e+01, 2.20010e+13],
       [2.43334e+05, 7.50000e+01, 2.20010e+13]], shape=(331733, 3))

In [18]:
X_train, X_test, Y_train, Y_test=split_scaler(kbest,dep_Y)   
    
        
classifier,Accuracy,report,X_test,Y_test,cm=logistic(X_train,Y_train,X_test)
acclog.append(Accuracy)

classifier,Accuracy,report,X_test,Y_test,cm=svm_linear(X_train,Y_train,X_test)  
accsvml.append(Accuracy)
    
classifier,Accuracy,report,X_test,Y_test,cm=svm_NL(X_train,Y_train,X_test)  
accsvmnl.append(Accuracy)
    
classifier,Accuracy,report,X_test,Y_test,cm=knn(X_train,Y_train,X_test)  
accknn.append(Accuracy)
    
classifier,Accuracy,report,X_test,Y_test,cm=Navie(X_train,Y_train,X_test)  
accnav.append(Accuracy)
    
classifier,Accuracy,report,X_test,Y_test,cm=Decision(X_train,Y_train,X_test)  
accdes.append(Accuracy)
    
classifier,Accuracy,report,X_test,Y_test,cm=random(X_train,Y_train,X_test)  
accrf.append(Accuracy)
    
result=selectk_Classification(acclog,accsvml,accsvmnl,accknn,accnav,accdes,accrf)

In [19]:
result #

,Logistic,SVMl,SVMnl,KNN,Navie,Decision,Random
ChiSquare,0.99994,0.99994,0.99994,0.99994,0.99994,0.999904,0.999916
